# 02. In-Memory Temporal Knowledge Graph Analysis

This notebook analyzes the graph topology, degree distributions, and event-reification pattern of the instantiated temporal knowledge graph using `InMemoryKGStore` (NetworkX-based).

In [ ]:
import sys
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config import settings
from src.knowledge_graph.memory_store import InMemoryKGStore

## 1. Loading the Processed Knowledge Graph

In [ ]:
store = InMemoryKGStore()
graph_path = settings.PROCESSED_DATA_DIR / "knowledge_graph.json"
print(f"Loading cached graph from {graph_path}...")
store.load(graph_path)

stats = store.get_statistics()
print("Graph Store Statistics:")
for k, v in stats.items():
    print(f"  {k}: {v:,}")

## 2. Graph Topologies & Reified Event Pattern

Events are modeled using the **Event Node Reification pattern**:
`(Entity: Actor) -> [:ACTOR] -> (Event) -> [:TARGET] -> (Entity: Target)`

In [ ]:
G = store.graph
print(f"Total NetworkX Nodes: {G.number_of_nodes():,}")
print(f"Total NetworkX Edges: {G.number_of_edges():,}")

entity_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "entity"]
event_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "event"]
print(f"Entity nodes: {len(entity_nodes)}")
print(f"Event nodes:  {len(event_nodes)}")

### Degree Distribution of Top Political Entities

In [ ]:
degrees = dict(G.degree())
entity_degrees = {G.nodes[n].get("name", n): degrees[n] for n in entity_nodes}
top_entities = sorted(entity_degrees.items(), key=lambda x: x[1], reverse=True)[:15]

df_top = pd.DataFrame(top_entities, columns=["Entity", "Degree"])
plt.figure(figsize=(10, 5))
plt.barh(df_top["Entity"][::-1], df_top["Degree"][::-1], color="teal")
plt.title("Top 15 Most Connected Entities in Knowledge Graph")
plt.xlabel("Degree (Event Connections)")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 3. Querying Entity Relationships Across Time

In [ ]:
sample_events = store.get_events_between_entities("United States", "Russia", limit=5)
print(f"Sample events between US and Russia:")
for ev in sample_events:
    print(f"  {ev.get('timestamp')}: {ev.get('subject')} --[{ev.get('predicate')}]--> {ev.get('object')}")